# 03 · Seuil, résultat sur le test, ROI, équité (axes 5, 6, 7, 9)
Le seuil est choisi par le coût sur les prédictions hors-échantillon du **train**. Le **test** n'est utilisé qu'une fois, pour le modèle et le seuil finaux.
Hypothèses (explicites, testées en sensibilité) : perte en cas de défaut = 60 % du montant, marge perdue sur un refus à tort = 12 %, 32 000 demandes/an, coût du projet 150 k$, 70 % du gain mesuré réalisé en production.

In [1]:
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             roc_auc_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_val_predict,
                                     cross_val_score, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

sys.path.insert(0, os.path.abspath("../src"))
from features import CAT_COLS, NUM_COLS, TARGET, add_features, clean, load_raw
from costs import confusion, metrics, roi, total_cost

warnings.filterwarnings("ignore")
RS = 42
ROOT = os.path.abspath("..")
FIG = os.path.join(ROOT, "reports", "figures")
os.makedirs(FIG, exist_ok=True)

# ---------------- hypothèses business (axes 6-7), toutes testées en sensibilité -------------
LGD, MARGIN = 0.60, 0.12          # perte si défaut = 60 % du montant ; marge nette perdue si refus = 12 %
N_ANNUAL = 32_000                 # nombre de demandes traitées par an (hypothèse = taille du jeu)
COUT_PROJET = 150_000             # 1re année : développement 80 k + intégration/validation 30 k + exploitation 40 k
GAIN_REALISE = 0.70               # décote : seule une part du gain mesuré hors échantillon est réalisée en production
GUARDRAIL_PRECISION = 0.50        # au plus 1 client sain refusé pour 1 défaut évité

NAVY, TEAL, ORANGE, GREY, RED = "#1F3A5F", "#2A9D8F", "#E76F51", "#8D99AE", "#C0392B"
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 10, "axes.titleweight": "bold", "figure.dpi": 150})
R = {}                            # tous les résultats
t0 = time.time()

def save(fig, name):
    fig.savefig(os.path.join(FIG, name), bbox_inches="tight"); plt.close(fig)

In [2]:
# ------------------------------------------------------------------ 1. données
raw = load_raw(os.path.join(ROOT, "data", "credit_risk_dataset.csv"))
R["raw"] = {"n": len(raw), "cols": raw.shape[1],
            "missing": raw.isna().sum()[lambda s: s > 0].to_dict(),
            "age_max": float(raw.person_age.max()), "emp_max": float(raw.person_emp_length.max()),
            "income_max": float(raw.person_income.max())}
df, qc = clean(raw)
R["clean"] = qc
df = add_features(df)
R["default_rate"] = float(df[TARGET].mean())
R["n_clean"] = len(df)
R["rate_by"] = {
    "grade": df.groupby("loan_grade")[TARGET].mean().round(3).to_dict(),
    "prior_default": df.groupby("cb_person_default_on_file")[TARGET].mean().round(3).to_dict(),
    "home": df.groupby("person_home_ownership")[TARGET].mean().round(3).to_dict(),
    "intent": df.groupby("loan_intent")[TARGET].mean().round(3).to_dict(),
}
df["lpi_bin"] = pd.qcut(df["loan_percent_income"], 8, duplicates="drop")
R["rate_by"]["lpi_bin_first_last"] = [float(df.groupby("lpi_bin", observed=True)[TARGET].mean().iloc[0]),
                                      float(df.groupby("lpi_bin", observed=True)[TARGET].mean().iloc[-1])]
R["missing_rate_by_default"] = {
    "int_rate_missing": df.assign(m=df.loan_int_rate.isna()).groupby("m")[TARGET].mean().round(3).to_dict()}

# ------------------------------------------------------------------ 2. split (AVANT toute préparation)
X = df[NUM_COLS + CAT_COLS]
y = df[TARGET]
idx_tr, idx_te = train_test_split(df.index, test_size=0.2, stratify=y, random_state=RS)
X_tr, X_te, y_tr, y_te = X.loc[idx_tr], X.loc[idx_te], y.loc[idx_tr], y.loc[idx_te]
amt_tr, amt_te = df.loc[idx_tr, "loan_amnt"].values, df.loc[idx_te, "loan_amnt"].values
R["split"] = {"train": len(X_tr), "test": len(X_te), "default_train": float(y_tr.mean()),
              "default_test": float(y_te.mean())}


def make_prep(num_cols, cat_cols):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                          ("sc", StandardScaler())]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])


def make_pipe(clf, num_cols=NUM_COLS, cat_cols=CAT_COLS):
    return Pipeline([("prep", make_prep(num_cols, cat_cols)), ("clf", clf)])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)

In [3]:
import joblib
d = joblib.load(os.path.join(ROOT, "reports", "model.joblib"))
model, chosen, table = d["model"], d["chosen"], d["table"]
print(chosen)

Gradient Boosting (HGB)


## Seuil par le coût (prédictions OOF du train)

In [4]:
# ------------------------------------------------------------------ 4. seuil par le coût (axe 6) -> OOF sur le TRAIN
oof = cross_val_predict(clone(model), X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
grid_t = np.round(np.arange(0.05, 0.91, 0.01), 2)
rows = []
for t in grid_t:
    ref = oof >= t
    m = metrics(y_tr.values, ref)
    m["t"] = float(t)
    m["cout"] = total_cost(y_tr.values, ref, amt_tr, LGD, MARGIN)
    rows.append(m)
thr = pd.DataFrame(rows)
t_cost_free = float(thr.loc[thr.cout.idxmin(), "t"])
feas = thr[thr.precision >= GUARDRAIL_PRECISION]
t_final = float(feas.loc[feas.cout.idxmin(), "t"])
R["thresholds"] = {
    "t_theorique": MARGIN / (MARGIN + LGD),
    "t_cout_min_sans_contrainte": t_cost_free, "t_final": t_final,
    "oof_table": {str(t): {k: float(thr.loc[thr.t == t, k].iloc[0]) for k in
                           ["precision", "recall", "f1", "refus", "cout"]}
                  for t in sorted({0.5, round(t_cost_free, 2), round(t_final, 2), 0.3})},
    "oof_cout_accorder_tout": total_cost(y_tr.values, np.zeros(len(y_tr), bool), amt_tr, LGD, MARGIN)}

In [5]:
# fig 5 : seuil
fig, axs = plt.subplots(1, 2, figsize=(9, 3.4))
ax = axs[0]
ax.plot(thr.t, thr.cout / 1e6, color=NAVY, lw=2)
ax.axvline(t_final, color=ORANGE, ls="--"); ax.axvline(0.5, color=GREY, ls=":")
ax.text(t_final + .01, ax.get_ylim()[1] * .92, f"retenu t={t_final:.2f}", color=ORANGE, fontsize=8)
ax.text(0.51, ax.get_ylim()[1] * .78, "défaut 0,5", color=GREY, fontsize=8)
ax.set_xlabel("seuil de refus (score ≥ seuil)"); ax.set_ylabel("coût d'erreur total, train OOF (M$)")
ax.set_title("Le seuil 0,5 coûte plus que le seuil retenu")
ax = axs[1]
ax.plot(thr.t, thr.precision, label="Precision", color=TEAL, lw=2); ax.plot(thr.t, thr.recall, label="Recall", color=ORANGE, lw=2)
ax.axhline(GUARDRAIL_PRECISION, color=GREY, ls=":"); ax.text(.45, GUARDRAIL_PRECISION + .02, "garde-fou Precision 50 %", fontsize=7, color=GREY)
ax.axvline(t_final, color=ORANGE, ls="--"); ax.set_xlabel("seuil"); ax.set_title("Compromis Precision / Recall"); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); save(fig, "fig5_threshold.png")

## Résultat final sur le test (une seule fois)

In [6]:
# ------------------------------------------------------------------ 5. TEST (utilisé une seule fois)
p_te = model.predict_proba(X_te)[:, 1]
ref_model = p_te >= t_final
ref_all = np.zeros(len(y_te), bool)
ref_rule = (X_te["grade_num"].values >= 4)          # règle actuelle supposée : refus des notes D à G
ref_t05 = p_te >= 0.5
yt = y_te.values
R["test"] = {
    "roc_auc": float(roc_auc_score(yt, p_te)), "pr_auc": float(average_precision_score(yt, p_te)),
    "baseline_pr_auc": float(yt.mean()),
    "modele": metrics(yt, ref_model), "seuil_05": metrics(yt, ref_t05),
    "regle_notes_DG": metrics(yt, ref_rule), "accorder_tout": metrics(yt, ref_all),
    "cout": {"modele": total_cost(yt, ref_model, amt_te, LGD, MARGIN),
             "seuil_05": total_cost(yt, ref_t05, amt_te, LGD, MARGIN),
             "regle": total_cost(yt, ref_rule, amt_te, LGD, MARGIN),
             "accorder_tout": total_cost(yt, ref_all, amt_te, LGD, MARGIN)},
    "n_defauts": int(yt.sum()), "n": int(len(yt))}
print("TEST", json.dumps(R["test"]["modele"]), flush=True)

TEST {"tp": 1182, "fp": 374, "fn": 236, "tn": 4690, "precision": 0.7596401028277635, "recall": 0.8335684062059239, "f1": 0.7948890383322126, "refus": 0.2400493674791731}


In [7]:
# fig 6 : test
fig, axs = plt.subplots(1, 2, figsize=(9, 3.6))
ax = axs[0]
tp, fp, fn, tn = confusion(yt, ref_model)
cmx = np.array([[tn, fp], [fn, tp]])
ax.imshow(cmx, cmap="Blues")
for i in range(2):
    for j in range(2): ax.text(j, i, f"{cmx[i, j]:,}".replace(",", " "), ha="center", va="center",
                               color="white" if cmx[i, j] > cmx.max() / 2 else "black", fontsize=13, fontweight="bold")
ax.set_xticks([0, 1]); ax.set_xticklabels(["accordé", "refusé"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["remboursé", "défaut"])
ax.set_xlabel("décision du modèle"); ax.set_ylabel("réalité"); ax.set_title(f"Matrice de confusion (test, seuil {t_final:.2f})")
ax = axs[1]
pr, rc, _ = precision_recall_curve(yt, p_te)
ax.plot(rc, pr, color=NAVY, lw=2, label=f"modèle (PR-AUC {R['test']['pr_auc']:.2f})")
ax.axhline(yt.mean(), color=GREY, ls="--", label=f"baseline ({yt.mean():.2f})")
mm = R["test"]["modele"]; ax.scatter([mm["recall"]], [mm["precision"]], color=ORANGE, zorder=5, s=50, label="point retenu")
rr_ = R["test"]["regle_notes_DG"]; ax.scatter([rr_["recall"]], [rr_["precision"]], color=RED, marker="s", zorder=5, s=45, label="règle actuelle (notes D-G)")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title("Courbe Precision-Recall sur le test"); ax.legend(frameon=False, fontsize=7, loc="center left", bbox_to_anchor=(0.0, 0.42))
fig.tight_layout(); save(fig, "fig6_test.png")

## ROI et analyse de sensibilité

In [8]:
# ------------------------------------------------------------------ 6. ROI + sensibilité (axe 7)
scale = N_ANNUAL / len(yt)


def annual_costs(lgd, margin, p=p_te, t=t_final):
    c_m = total_cost(yt, p >= t, amt_te, lgd, margin) * scale
    c_r = total_cost(yt, ref_rule, amt_te, lgd, margin) * scale
    c_a = total_cost(yt, ref_all, amt_te, lgd, margin) * scale
    return c_m, c_r, c_a


rng = np.random.default_rng(RS)
p_noisy = np.clip(p_te + rng.normal(0, 0.15, len(p_te)), 0, 1)
scen_def = [("Pessimiste", 0.40, 0.15, 250_000, p_te, 0.40),
            ("Central", LGD, MARGIN, COUT_PROJET, p_te, GAIN_REALISE),
            ("Optimiste", 0.75, 0.10, 120_000, p_te, 1.00),
            ("Dérive (scores bruités)", LGD, MARGIN, COUT_PROJET, p_noisy, GAIN_REALISE)]
scen = []
for nm, lgd, mg, cp, p, frac in scen_def:
    c_m, c_r, c_a = annual_costs(lgd, mg, p=p)
    g_mesure = c_r - c_m
    g = g_mesure * frac
    r_, back = roi(g, cp)
    scen.append({"scenario": nm, "lgd": lgd, "marge": mg, "cout_projet": cp, "part_realisee": frac,
                 "cout_sans": c_r, "cout_avec": c_m, "gain_mesure": g_mesure, "gain": g, "roi": r_,
                 "retour_mois": back, "gain_vs_accorder_tout": c_a - c_m,
                 "gain_mesure_par_1000_demandes": g_mesure / N_ANNUAL * 1000})
R["roi"] = scen
lgds = [0.30, 0.45, 0.60, 0.75, 0.90]
margins = [0.06, 0.09, 0.12, 0.15, 0.18]
sens = np.array([[annual_costs(l, m)[1] - annual_costs(l, m)[0] for m in margins] for l in lgds])
R["sens"] = {"lgds": lgds, "margins": margins, "gain": sens.tolist()}
# part minimale du gain mesuré qui doit se réaliser pour rembourser le projet (scénario central)
g_central = [x for x in scen if x["scenario"] == "Central"][0]["gain_mesure"]
R["break_even_part_realisee"] = COUT_PROJET / g_central
R["roi_selon_part_realisee"] = {str(f): roi(g_central * f, COUT_PROJET)[0] for f in (1.0, 0.7, 0.4, 0.1, 0.05)}

In [9]:
import pandas as pd
pd.DataFrame(R["roi"]).round(2)

,scenario,lgd,marge,cout_projet,part_realisee,cout_sans,cout_avec,gain_mesure,gain,roi,retour_mois,gain_vs_accorder_tout,gain_mesure_par_1000_demandes
0,Pessimiste,0.40,0.15,250000,0.4,20382820.12,6812545.51,13570274.61,5428109.84,2071.24,0.55,23860209.81,424071.08
1,Central,0.60,0.12,150000,0.7,28202377.04,8386952.18,19815424.87,13870797.41,9147.20,0.13,37622180.81,619232.03
2,Optimiste,0.75,0.10,120000,1.0,34123517.43,9611373.03,24512144.40,24512144.40,20326.79,0.06,47900043.20,766004.51
3,Dérive (scores bruités),0.60,0.12,150000,0.7,28202377.04,11865049.06,16337327.99,11436129.59,7524.09,0.16,34144083.92,510541.50


In [10]:
# fig 8 : sensibilité ROI
fig, ax = plt.subplots(figsize=(6, 3.6))
im = ax.imshow(sens / 1e6, cmap="YlGn", aspect="auto", origin="lower")
ax.set_xticks(range(len(margins))); ax.set_xticklabels([f"{m:.0%}" for m in margins])
ax.set_yticks(range(len(lgds))); ax.set_yticklabels([f"{l:.0%}" for l in lgds])
for i in range(len(lgds)):
    for j in range(len(margins)): ax.text(j, i, f"{sens[i, j]/1e6:.2f}", ha="center", va="center", fontsize=8, color="white" if sens[i, j] > 0.6 * sens.max() else "black")
ax.set_xlabel("marge nette perdue sur un refus à tort"); ax.set_ylabel("perte en cas de défaut (LGD)")
ax.set_title("Gain annuel mesuré vs règle actuelle (M$, avant décote)"); fig.colorbar(im, shrink=.8); save(fig, "fig8_sens.png")

## Importance des variables, équité, ablations, erreurs

In [11]:
# ------------------------------------------------------------------ 7. diagnostics, équité, ablations
perm = permutation_importance(model, X_te, y_te, scoring="average_precision", n_repeats=5,
                              random_state=RS, n_jobs=1)
imp = pd.Series(perm.importances_mean, index=X_te.columns).sort_values(ascending=False)
R["perm_importance"] = imp.round(4).head(10).to_dict()

age = X_te["person_age"].values
groups = {"Âge < 25": age < 25, "Âge 25-29": (age >= 25) & (age < 30), "Âge ≥ 30": age >= 30,
          "Locataire": X_te.person_home_ownership.values == "RENT",
          "Hypothèque": X_te.person_home_ownership.values == "MORTGAGE",
          "Propriétaire": X_te.person_home_ownership.values == "OWN"}
fair = {}
for g, m in groups.items():
    yy, rr = yt[m], ref_model[m]
    tp, fp, fn, tn = confusion(yy, rr)
    fair[g] = {"n": int(m.sum()), "taux_defaut": float(yy.mean()), "taux_refus": float(rr.mean()),
               "recall": tp / (tp + fn) if tp + fn else 0.0, "fpr": fp / (fp + tn) if fp + tn else 0.0}
R["fairness"] = fair

abl = {}
base_cols = (NUM_COLS, CAT_COLS)
variants = {"Modèle complet": (NUM_COLS, CAT_COLS),
            "Sans l'âge": ([c for c in NUM_COLS if c not in ("person_age", "emp_to_age")], CAT_COLS),
            "Sans note interne ni taux (grade, loan_int_rate)":
                ([c for c in NUM_COLS if c not in ("grade_num", "loan_int_rate")], CAT_COLS)}
for nm, (nc, cc) in variants.items():
    est = clone(model)
    est = make_pipe(clone(model.named_steps["clf"]), nc, cc)
    s = cross_val_score(est, X_tr[nc + cc], y_tr, cv=cv, scoring="average_precision")
    a = cross_val_score(est, X_tr[nc + cc], y_tr, cv=cv, scoring="roc_auc")
    abl[nm] = {"ap": float(s.mean()), "ap_std": float(s.std()), "auc": float(a.mean())}
R["ablation"] = abl

# erreurs typiques : profil moyen des FN (défauts non détectés) vs TP
dte = df.loc[idx_te].copy()
dte["p"] = p_te
fn_mask = (yt == 1) & ~ref_model
tp_mask = (yt == 1) & ref_model
R["errors"] = {
    "fn_n": int(fn_mask.sum()), "tp_n": int(tp_mask.sum()),
    "fn_grade_mean": float(dte.loc[fn_mask, "grade_num"].mean()),
    "tp_grade_mean": float(dte.loc[tp_mask, "grade_num"].mean()),
    "fn_lpi_mean": float(dte.loc[fn_mask, "loan_percent_income"].mean()),
    "tp_lpi_mean": float(dte.loc[tp_mask, "loan_percent_income"].mean()),
    "fn_amnt_mean": float(dte.loc[fn_mask, "loan_amnt"].mean()),
    "tp_amnt_mean": float(dte.loc[tp_mask, "loan_amnt"].mean()),
    "fn_prior_default_share": float((dte.loc[fn_mask, "cb_person_default_on_file"] == "Y").mean()),
    "tp_prior_default_share": float((dte.loc[tp_mask, "cb_person_default_on_file"] == "Y").mean())}

In [12]:
# fig 7 : importance
fig, ax = plt.subplots(figsize=(6.2, 3.2))
top = imp.head(8)[::-1]
nice = {"grade_num": "note interne", "loan_percent_income": "montant / revenu", "loan_int_rate": "taux d'intérêt",
        "person_home_ownership": "logement", "log_income": "revenu (log)", "cb_person_default_on_file": "défaut passé",
        "loan_intent": "objet du prêt", "person_emp_length": "ancienneté emploi", "loan_amnt": "montant",
        "person_age": "âge", "emp_to_age": "ancienneté / âge", "cb_person_cred_hist_length": "historique crédit"}
ax.barh([nice.get(i, i) for i in top.index], top.values, color=TEAL)
ax.set_xlabel("baisse de PR-AUC si la variable est mélangée (test)")
ax.set_title("Variables qui comptent le plus pour le modèle"); save(fig, "fig7_importance.png")

In [13]:
print(R["perm_importance"]); print(R["ablation"]); print(R["errors"])
pd.DataFrame(R["fairness"]).T.round(3)

{'loan_percent_income': 0.3138, 'person_home_ownership': 0.1991, 'grade_num': 0.1716, 'log_income': 0.1534, 'loan_intent': 0.0897, 'person_age': 0.0181, 'person_emp_length': 0.0112, 'loan_int_rate': 0.0053, 'loan_amnt': 0.0024, 'emp_to_age': 0.0024}
{'Modèle complet': {'ap': 0.8993217444491315, 'ap_std': 0.002723188722218434, 'auc': 0.9438075000680307}, "Sans l'âge": {'ap': 0.8941833992063384, 'ap_std': 0.0021710923599171476, 'auc': 0.9409030290424649}, 'Sans note interne ni taux (grade, loan_int_rate)': {'ap': 0.8080889160049141, 'ap_std': 0.007050978214192315, 'auc': 0.8948191908015749}}
{'fn_n': 236, 'tp_n': 1182, 'fn_grade_mean': 2.152542372881356, 'tp_grade_mean': 3.22419627749577, 'fn_lpi_mean': 0.14542372881355933, 'tp_lpi_mean': 0.2667343485617597, 'fn_amnt_mean': 9002.860169491525, 'tp_amnt_mean': 11343.654822335026, 'fn_prior_default_share': 0.17372881355932204, 'tp_prior_default_share': 0.3544839255499154}


,n,taux_defaut,taux_refus,recall,fpr
Âge < 25,2380.0,0.228,0.246,0.851,0.067
Âge 25-29,2257.0,0.224,0.246,0.824,0.079
Âge ≥ 30,1845.0,0.201,0.225,0.822,0.075
Locataire,3275.0,0.314,0.346,0.890,0.097
Hypothèque,2651.0,0.129,0.142,0.650,0.067
Propriétaire,533.0,0.079,0.077,0.952,0.002
